In [ ]:
import os, json, re
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
from pycocotools.coco import COCO

from sklearn.metrics import precision_recall_fscore_support

from transformers import BitsAndBytesConfig

In [ ]:
COCO_ROOT = "/BS/generative_modelling_for_image_understanding/nobackup/data/DETECTRON2_DATASETS/coco"
SPLIT = "val2017"  # val2017 or train2017

ANN_FILE = os.path.join(COCO_ROOT, "annotations", f"instances_{SPLIT}.json")
IMG_DIR  = os.path.join(COCO_ROOT, SPLIT)

assert os.path.isfile(ANN_FILE), ANN_FILE
assert os.path.isdir(IMG_DIR), IMG_DIR

device = "cuda" if torch.cuda.is_available() else "cpu"
device


In [ ]:
coco = COCO(ANN_FILE)

cat_ids = coco.getCatIds()
cats = coco.loadCats(cat_ids)
cats_sorted = sorted(cats, key=lambda x: x["id"])

cat_ids_sorted = [c["id"] for c in cats_sorted]
COCO_LABELS = [c["name"] for c in cats_sorted]  # official COCO category names

catid_to_index = {cid: i for i, cid in enumerate(cat_ids_sorted)}
label_to_idx = {name: i for i, name in enumerate(COCO_LABELS)}

print("COCO initialized.")
print("Num classes:", len(COCO_LABELS))
print("First 15:", COCO_LABELS[:15])

def get_gt_labels_for_img(img_id: int):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    present_cids = sorted({a["category_id"] for a in anns})
    return [coco.loadCats([cid])[0]["name"] for cid in present_cids]


In [ ]:
# model_id = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
# dtype = torch.float16 if device == "cuda" else torch.float32

# model = LlavaOnevisionForConditionalGeneration.from_pretrained(
#     model_id,
#     torch_dtype=dtype,
#     low_cpu_mem_usage=True,
# ).to(device).eval()

# processor = AutoProcessor.from_pretrained(model_id)

# # keep this
# processor.tokenizer.padding_side = "left"

# if processor.tokenizer.pad_token_id is None:
#     processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# print("Loaded model:", model_id)
# print("Device:", device, "| dtype:", dtype)
# print("pad_token_id:", processor.tokenizer.pad_token_id, "eos_token_id:", processor.tokenizer.eos_token_id)

model_id = "llava-hf/llava-onevision-qwen2-7b-ov-hf"

# ---- recommended knobs for 7B ----
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# Optional: use FlashAttention if installed (often helps speed/mem)
# from transformers import BitsAndBytesConfig  # only if you do quantization

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )

# model = LlavaOnevisionForConditionalGeneration.from_pretrained(
#     model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     low_cpu_mem_usage=True,
# ).eval()

model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    device_map="auto" if device == "cuda" else None,   # <<< important for 7B
).eval()

processor = AutoProcessor.from_pretrained(model_id)

# keep this
processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

print("Loaded model:", model_id)
print("Device:", device, "| dtype:", dtype)
print("pad_token_id:", processor.tokenizer.pad_token_id, "eos_token_id:", processor.tokenizer.eos_token_id)


In [ ]:
def show_image(path, title=None, figsize=(8,5)):
    img = Image.open(path).convert("RGB")
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()
    return img


In [ ]:
def norm_phrase(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def singularize_basic(w: str) -> str:
    w = w.strip().lower()
    if w == "knives":
        return "knife"
    if w == "mice":
        return "mouse"
    if w.endswith("ies") and len(w) > 3:
        return w[:-3] + "y"
    if w.endswith("ves") and len(w) > 3:
        return w[:-3] + "f"
    if w.endswith("s") and not w.endswith("ss") and len(w) > 3:
        return w[:-1]
    return w

def clean_items(items):
    cleaned = []
    for it in items:
        it = norm_phrase(it)
        it = singularize_basic(it)
        if len(it) <= 1:
            continue
        cleaned.append(it)
    return cleaned


In [ ]:
# >>> CHANGE: do NOT collect dict keys anymore.
# Only collect string VALUES (this prevents "categories/name/label/room" pollution).
# This change is very low-risk: it doesn't affect "quoted"/"fallback" mode at all,
# and makes "json_any" behave sanely when the model emits schema JSON.

def _collect_string_values_from_json(x, out):
    if isinstance(x, str):
        out.append(x)
    elif isinstance(x, list):
        for v in x:
            _collect_string_values_from_json(v, out)
    elif isinstance(x, dict):
        for v in x.values():
            _collect_string_values_from_json(v, out)

'''
def parse_objects_any(text: str):
    """
    Parse objects from:
      - JSON list: ["toilet","sink"]
      - JSON dict: {"toilet":"toilet","sink":"sink", ...}
      - nested JSON
      - truncated outputs: extracts quoted strings
      - fallback split
    Returns: (items, parse_mode)
    """
    t = text.strip()
    t = re.sub(r"^```(json)?\s*", "", t)
    t = re.sub(r"\s*```$", "", t)

    candidates = []
    s1, e1 = t.find("["), t.rfind("]")
    if s1 != -1 and e1 != -1 and e1 > s1:
        candidates.append(t[s1:e1+1])

    s2, e2 = t.find("{"), t.rfind("}")
    if s2 != -1 and e2 != -1 and e2 > s2:
        candidates.append(t[s2:e2+1])

    for c in candidates:
        try:
            obj = json.loads(c)
            out = []
            _collect_string_values_from_json(obj, out)
            out = [o.strip().lower() for o in out if isinstance(o, str)]
            return out, "json_any"
        except Exception:
            pass

    quoted = re.findall(r'"([^"]+)"', t)
    if quoted:
        return [q.strip().lower() for q in quoted], "quoted"

    t2 = re.sub(r"[\[\]\{\}\"]", " ", t)
    parts = re.split(r"[,;\n]+", t2)
    items = []
    for p in parts:
        p = p.strip().lower()
        if not p:
            continue
        if len(p.split()) > 4:
            continue
        items.append(p)
    return items, "fallback"
'''
def parse_objects_any(text: str):
    t = text.strip()
    t = re.sub(r"^```(json)?\s*", "", t)
    t = re.sub(r"\s*```$", "", t)

    # 1) try parsing ALL json arrays
    arrays = re.findall(r"\[[\s\S]*?\]", t)  # non-greedy, multiple blocks
    merged = []
    for a in arrays:
        try:
            obj = json.loads(a)
            out = []
            _collect_strings_from_json(obj, out)
            merged.extend(out)
        except Exception:
            pass
    if merged:
        return [str(x).strip().lower() for x in merged if isinstance(x, str)], "json_any_multi"

    # 2) try parsing ALL json dicts
    dicts = re.findall(r"\{[\s\S]*?\}", t)
    merged = []
    for d in dicts:
        try:
            obj = json.loads(d)
            out = []
            _collect_strings_from_json(obj, out)
            merged.extend(out)
        except Exception:
            pass
    if merged:
        return [str(x).strip().lower() for x in merged if isinstance(x, str)], "json_any_multi"

    # 3) fallback quoted
    quoted = re.findall(r'"([^"]+)"', t)
    if quoted:
        return [q.strip().lower() for q in quoted], "quoted"

    # 4) last fallback split
    t2 = re.sub(r"[\[\]\{\}\"]", " ", t)
    parts = re.split(r"[,;\n]+", t2)
    items = []
    for p in parts:
        p = p.strip().lower()
        if not p:
            continue
        if len(p.split()) > 4:
            continue
        items.append(p)
    return items, "fallback"


In [ ]:
COCO_SET = {norm_phrase(x): x for x in COCO_LABELS}

# >>> CHANGE: your latest alias dict exactly as provided
ALIAS_TO_COCO = {
    "sofa": "couch",
    "settee": "couch",
    "television": "tv",
    "mobile phone": "cell phone",
    "phone": "cell phone",
    "smartphone": "cell phone",
    "cellphone": "cell phone",
    "bike": "bicycle",
    "motorbike": "motorcycle",
    "mug": "cup",
    "teddy": "teddy bear",
    "stuffed bear": "teddy bear",
    "stuffed toy": "teddy bear",
    "traffic signal": "traffic light",
    "signal": "traffic light",
    # plural safety
    "knives": "knife",
    "forks": "fork",
    "spoons": "spoon",
    "cups": "cup",
    "bottles": "bottle",
}

# SAFE structural aliases (low-risk)
ALIAS_TO_COCO.update({
    "table": "dining table",
    "wooden table": "dining table",
    "kitchen counter": "dining table",
    "counter": "dining table",
    "stove": "oven",
    "stove top": "oven",
})

def map_to_coco(objects):
    mapped = set()
    unmapped = []

    for obj in objects:
        o = norm_phrase(obj)
        if not o:
            continue

        if o in COCO_SET:
            mapped.add(COCO_SET[o])
            continue

        if o in ALIAS_TO_COCO:
            mapped.add(ALIAS_TO_COCO[o])
            continue

        # substring alias match (keep alias list small)
        hit_alias = None
        for alias, coco_lab in ALIAS_TO_COCO.items():
            #if alias in o:
            # strict word-boundary substring match
            if re.search(rf"\b{re.escape(alias)}\b", o):
                hit_alias = coco_lab
                break
        if hit_alias is not None:
            mapped.add(hit_alias)
            continue

        # multiword COCO contains match
        hit = None
        for coco_norm, coco_official in COCO_SET.items():
            if " " in coco_norm and coco_norm in o:
                hit = coco_official
                break

        if hit is not None:
            mapped.add(hit)
        else:
            unmapped.append(obj)

    return sorted(mapped), unmapped


In [ ]:
def build_prompt_main_objects():
    # Important: text then image
    conversation = [{
        "role": "user",
        "content": [
            {"type": "text", "text": (
                "List the object categories that are actually visible in the image.\n"
                "Return ONLY a JSON array of at most 12 UNIQUE short nouns (1-2 words).\n"
                "Do NOT include example words from the prompt unless they are truly visible.\n"
                "If you are unsure, omit the object.\n"
                "No explanations. No extra text."
            )},
            {"type": "image"},
        ],
    }]
    return processor.apply_chat_template(conversation, add_generation_prompt=True)

def build_prompt_small_objects():
    # Stricter + anti-guess
    conversation = [{
        "role": "user",
        "content": [
            {"type": "text", "text": (
                "List ONLY small objects that are CLEARLY visible (utensils/containers/handheld items).\n"
                "Return ONLY a JSON array of at most 8 UNIQUE short nouns (1-2 words).\n"
                "If you are not 100% sure an object is present, DO NOT include it.\n"
                "Do NOT guess utensils.\n"
                "No explanations. No extra text."
            )},
            {"type": "image"},
        ],
    }]
    return processor.apply_chat_template(conversation, add_generation_prompt=True)


In [ ]:
# @torch.no_grad()
# def generate_with_prompt(image: Image.Image, prompt: str, max_new_tokens=96, repetition_penalty=1.2):
#     inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
#     out = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         repetition_penalty=repetition_penalty,
#         pad_token_id=processor.tokenizer.pad_token_id,
#     )
#     gen_only = out[0, inputs["input_ids"].shape[1]:]
#     return processor.decode(gen_only, skip_special_tokens=True).strip()


@torch.no_grad()
def generate_with_prompt(image: Image.Image, prompt: str, max_new_tokens=96, repetition_penalty=1.2):
    inputs = processor(images=image, text=prompt, return_tensors="pt")

    # If model is fully on one device, we can move inputs.
    # With device_map="auto", Accelerate will handle device placement,
    # but inputs must be on the "first" device if model expects it.
    if hasattr(model, "device"):
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=repetition_penalty,
        pad_token_id=processor.tokenizer.pad_token_id,
    )

    gen_only = out[0, inputs["input_ids"].shape[1]:]
    return processor.decode(gen_only, skip_special_tokens=True).strip()


In [ ]:
def predict_single_pass(image: Image.Image, max_new_tokens=96, repetition_penalty=1.2):
    prompt = build_prompt_main_objects()
    raw = generate_with_prompt(image, prompt, max_new_tokens=max_new_tokens, repetition_penalty=repetition_penalty)

    items, mode = parse_objects_any(raw)
    items = sorted(set(clean_items(items)))

    mapped, unmapped = map_to_coco(items)
    return {
        "mapped": mapped,
        "unmapped": unmapped,
        "items": items,
        "parse_modes": [mode],
        "raws": [raw],
        "prompts": [prompt],
    }

def predict_two_pass_gated(
    image: Image.Image,
    max_new_tokens_main=96,
    max_new_tokens_small=64,
    repetition_penalty=1.2,
    small_cap=4,
):
    p1 = build_prompt_main_objects()
    p2 = build_prompt_small_objects()

    raw1 = generate_with_prompt(image, p1, max_new_tokens=max_new_tokens_main, repetition_penalty=repetition_penalty)
    raw2 = generate_with_prompt(image, p2, max_new_tokens=max_new_tokens_small, repetition_penalty=repetition_penalty)

    items1, m1 = parse_objects_any(raw1)
    items2, m2 = parse_objects_any(raw2)

    items1 = sorted(set(clean_items(items1)))
    items2 = sorted(set(clean_items(items2)))

    mapped1, _ = map_to_coco(items1)
    mapped2, _ = map_to_coco(items2)

    anchors = {"person", "dining table", "toilet", "sink", "bed", "couch", "chair"}
    if len(set(mapped1) & anchors) == 0:
        mapped = mapped1
    else:
        mapped = sorted(set(mapped1) | set(mapped2[:max(0, small_cap)]))

    items = sorted(set(items1 + items2))
    return {
        "mapped": mapped,
        "items": items,
        "parse_modes": [m1, m2],
        "raws": [raw1, raw2],
        "prompts": [p1, p2],
        "mapped_main": mapped1,
        "mapped_small": mapped2,
    }


In [ ]:
def quick_caption(img_id):
    info = coco.loadImgs(img_id)[0]
    path = os.path.join(IMG_DIR, info["file_name"])
    image = Image.open(path).convert("RGB")

    conversation = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the image in one short sentence."},
            {"type": "image"},
        ],
    }]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    raw = generate_with_prompt(image, prompt, max_new_tokens=48, repetition_penalty=1.1)
    print(img_id, "->", raw)

for iid in coco.getImgIds()[:3]:
    quick_caption(iid)


In [ ]:
img_id = 397133
info = coco.loadImgs(img_id)[0]
path = os.path.join(IMG_DIR, info["file_name"])

print("Image ID:", img_id, "| file:", info["file_name"])
image = show_image(path, title=f"img_id={img_id}")

gt = get_gt_labels_for_img(img_id)
print("\nGT labels:", gt)

out1 = predict_single_pass(image)
out2 = predict_two_pass_gated(image)

print("\n=== Baseline 1 (single-pass) ===")
print("Parse modes:", out1["parse_modes"])
print("Cleaned items:", out1["items"])
print("Mapped COCO:", out1["mapped"])
print("Unmapped:", out1["unmapped"][:20])
print("Raw:\n", out1["raws"][0])

print("\n=== Baseline 2 (two-pass gated) ===")
print("Parse modes:", out2["parse_modes"])
print("Cleaned items:", out2["items"])
print("Mapped main:", out2["mapped_main"])
print("Mapped small:", out2["mapped_small"])
print("Final mapped:", out2["mapped"])
print("Raw main:\n", out2["raws"][0])
print("Raw small:\n", out2["raws"][1])


In [ ]:
def gt_multihot(img_ids_subset):
    Y = np.zeros((len(img_ids_subset), len(COCO_LABELS)), dtype=np.int32)
    for i, iid in enumerate(img_ids_subset):
        ann_ids = coco.getAnnIds(imgIds=[iid])
        anns = coco.loadAnns(ann_ids)
        present = {a["category_id"] for a in anns}
        for cid in present:
            if cid in catid_to_index:
                Y[i, catid_to_index[cid]] = 1
    return Y


In [ ]:
@torch.no_grad()
def pred_multihot(img_ids_subset, mode="single", verbose_every=0):
    Y = np.zeros((len(img_ids_subset), len(COCO_LABELS)), dtype=np.int32)
    mapped_lists = []
    raws_lists = []

    for i, iid in enumerate(img_ids_subset):
        info = coco.loadImgs(iid)[0]
        path = os.path.join(IMG_DIR, info["file_name"])
        image = Image.open(path).convert("RGB")

        if mode == "single":
            out = predict_single_pass(image)
        elif mode == "two_gated":
            out = predict_two_pass_gated(image)
        else:
            raise ValueError(mode)

        mapped = out["mapped"]
        for lab in mapped:
            Y[i, label_to_idx[lab]] = 1

        mapped_lists.append(mapped)
        raws_lists.append(out["raws"])

        if verbose_every and (i % verbose_every == 0):
            print(f"[{mode}] {i}/{len(img_ids_subset)} img_id={iid} mapped={mapped}")

    return Y, mapped_lists, raws_lists

def report_metrics(Y_true, Y_pred, name=""):
    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(Y_true, Y_pred, average="micro", zero_division=0)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(Y_true, Y_pred, average="macro", zero_division=0)
    print(f"\n=== {name} ===")
    print(f"Micro P/R/F1: {p_micro:.4f} / {r_micro:.4f} / {f1_micro:.4f}")
    print(f"Macro P/R/F1: {p_macro:.4f} / {r_macro:.4f} / {f1_macro:.4f}")


In [ ]:
K = 30
subset = coco.getImgIds()[:K]

Y_true = gt_multihot(subset)

Y_pred1, mapped1, raws1 = pred_multihot(subset, mode="single", verbose_every=10)
Y_pred2, mapped2, raws2 = pred_multihot(subset, mode="two_gated", verbose_every=10)

report_metrics(Y_true, Y_pred1, name="Baseline 1 (single-pass)")
report_metrics(Y_true, Y_pred2, name="Baseline 2 (two-pass gated)")


In [ ]:
def inspect_idx(idx):
    iid = subset[idx]
    info = coco.loadImgs(iid)[0]
    path = os.path.join(IMG_DIR, info["file_name"])
    show_image(path, title=f"idx={idx}, img_id={iid}")

    gt = get_gt_labels_for_img(iid)
    print("GT:", gt)

    print("\n[Single-pass] mapped:", mapped1[idx])
    print("Raw:", raws1[idx][0])

    print("\n[Two-pass gated] mapped:", mapped2[idx])
    print("Raw main:", raws2[idx][0])
    print("Raw small:", raws2[idx][1])

inspect_idx(0)
inspect_idx(1)
